# Triage severity model

Classifies a free-text symptom description into five levels, from self-care to
emergency. The text arrives in Bangla, romanised Banglish, English or a mix of
all three, which is how people actually write.

**The safety rule that matters:** the model's answer is never the final word.
A deterministic red-flag layer runs afterwards and can only ever raise the
severity. If the model calls chest pain with breathlessness routine, the rules
override it to emergency. The reverse cannot happen.

In [ ]:
import subprocess, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
print("project root:", ROOT)

## 1. Build the corpus

Two sources are combined:

* **Generated** — 9,000 notes assembled from the symptom lexicon, with the
  label derived from symptom acuity, duration, qualifier and age.
* **Real presentations** — 12,466 Bangla and Banglish notes built from 9,446
  genuine emergency department visits in the US CDC survey. Each recorded
  complaint is routed through the lexicon to its canonical symptom, then
  written back out in natural Bangla.

The English text of those visits is deliberately **not** trained on. The survey
asks how fast someone already inside an emergency department must be seen;
this model answers what a person at home should do. Mixing the two scales cost
15 points of macro-F1 when it was tried.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "build_bangla_from_real.py")], check=True)
subprocess.run([sys.executable, str(ROOT / "ml" / "build_triage_corpus.py")], check=True)

In [ ]:
import pandas as pd

corpus = pd.read_csv(ROOT / "data" / "triage" / "symptom_triage_dataset.csv")
print(f"{len(corpus):,} rows")
display(corpus.groupby(["source", "triage_level"]).size().unstack(fill_value=0))
display(corpus["language"].value_counts())
corpus.sample(5, random_state=0)[["text", "language", "triage_level", "source"]]

## 2. Features

A TF-IDF branch over words and another over character n-grams, joined to a
block of structured clinical features. Character n-grams matter more than usual
here: Banglish has no fixed spelling, so `buke betha`, `bukey byatha` and
`buk betha` must all reach the same place.

Age is a separate feature because it arrives as a form field, not in the text —
a text-only model is blind to it, and age changes the answer.

In [ ]:
from app.ai.features import clinical_feature_matrix, text_column

sample = corpus[["text", "age"]].head(3)
matrix = clinical_feature_matrix(sample)
print("structured features per row:", matrix.shape[1])

## 3. Train

Three candidates are compared on the validation split and the best by macro-F1
is kept. Macro-F1 rather than accuracy because the classes are very unbalanced
and getting the rare emergencies right is the whole point.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "train_triage_model.py")], check=True)

In [ ]:
metrics = json.loads((ROOT / "backend/app/ai/artifacts/triage_metrics.json").read_text())
print(json.dumps(metrics, indent=2))

## 4. Does it actually escalate an emergency?

The metric that matters is not accuracy. It is whether a genuine emergency,
written the way a frightened person writes it, reaches level 5.

In [ ]:
from app.ai.triage_service import triage

cases = [
    ("বুকে ব্যথা, শ্বাস নিতে কষ্ট", 55, "cardiac, Bangla"),
    ("bukey betha, ghamtesi", 55, "cardiac, Banglish"),
    ("রক্তবমি হচ্ছে", 45, "GI bleed"),
    ("খিঁচুনি হচ্ছে", 25, "seizure"),
    ("niswas nite parchi na", 60, "cannot breathe"),
    ("হালকা সর্দি কাশি", 25, "common cold — must NOT escalate"),
    ("পেট ব্যথা", 30, "one ordinary symptom — must NOT escalate"),
]

for text, age, label in cases:
    result = triage(text, age=age)
    print(f"L{result['severity_level']}  {label:42} {text[:34]}")

## Limitation

The Bangla training text is still generated, even where the underlying
presentation is real. No public dataset carries a clinician-assigned urgency
label against Bangla free text; that has to be collected with a hospital
partner under ethics approval. Until then this is decision support for review,
not a tool to be pointed at patients.